![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 01 | Exploratory Data Analysis

*Project 1 — SQL: From Data to Insight*

**Team:** Alex Russell  
**Dataset:** Inside Airbnb - Barcelona
---

Understanding your data before you design anything. You cannot decide which tables you need until you know which columns repeat, which are broken, and which identify a row.

> **Done when:** you can name the 3+ tables you are going to build, you have written down at least 2 research questions, and you know every data-quality problem notebook 02 will have to fix.

The sections below are a suggested route, not a form to fill in. Add sections, drop them, reorder them — what is graded is the analysis, not whether you kept the scaffold.

---
## 0. Setup

In [3]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.functions import *

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

---
## 1. Load the raw data

Run `python download_data.py <payments|airbnb|retail>` from the repo root first. Your files land in `data/raw/`, which is gitignored.

In [4]:
from pathlib import Path

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"

listings = pd.read_csv(RAW / "listings.csv.gz")
reviews = pd.read_csv(RAW / "reviews.csv.gz")

---
## 2. First look

Shape, column types, a handful of rows. What is one row of this data?

In [5]:
print("Listings shape:", listings.shape)
print("Reviews shape:", reviews.shape)

Listings shape: (15293, 90)
Reviews shape: (1033523, 6)


In [6]:
listings.info()

<class 'pandas.DataFrame'>
RangeIndex: 15293 entries, 0 to 15292
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            15293 non-null  int64  
 1   listing_url                                   15293 non-null  str    
 2   scrape_id                                     15293 non-null  int64  
 3   last_scraped                                  15293 non-null  str    
 4   source                                        15293 non-null  str    
 5   name                                          15293 non-null  str    
 6   description                                   14644 non-null  str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   15293 non-null  str    
 9   host_id                                       15293 non-null  int64  
 1

In [7]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 1033523 entries, 0 to 1033522
Data columns (total 6 columns):
 #   Column         Non-Null Count    Dtype
---  ------         --------------    -----
 0   listing_id     1033523 non-null  int64
 1   id             1033523 non-null  int64
 2   date           1033523 non-null  str  
 3   reviewer_id    1033523 non-null  int64
 4   reviewer_name  1033521 non-null  str  
 5   comments       1033409 non-null  str  
dtypes: int64(3), str(3)
memory usage: 311.7 MB


In [8]:
listings.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,price_quote_checkin_date,price_quote_checkout_date,price_quote_total_price,price_quote_price_per_night,price_quote_raw,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,18674,https://www.airbnb.com/rooms/18674,20260624162917,2026-06-25,city scrape,Huge flat for 8 people close to Sagrada Familia,110m2 apartment to rent in Barcelona. Located ...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,71615,https://www.airbnb.com/users/show/71615,1.462508e+18,https://www.airbnb.com/users/profile/146250835...,Maria Y Mireia Barcelona4Seasons,NaN,16.0,5.0,15.0,5.0,"Barcelona, Spain","We are Mireia and Maria, two multilingual entr...",NaN,NaN,NaN,f,NaN,https://a0.muscache.com/im/pictures/user/User/...,NaN,31.0,NaN,NaN,t,t,NaN,la Sagrada Família,Eixample,41.405560,2.17262,Entire rental unit,Entire home/apt,8,2.0,2 baths,3.0,5.0,"[""AC - split type ductless system"", ""Private p...",$409.00,2026-06-28,2026-06-29,409.00,409.00,"{""quote"": {""taxes"": null, ""currency"": ""EUR"", ""...",1.0,999.0,1.0,4.0,999.0,999.0,2.2,999.0,NaN,t,15,22,52,123,2026-06-25,59,9,1,121,9,54,22086.0,2013-05-27,2026-05-28,4.41,4.47,4.59,4.66,4.64,4.83,4.38,Spain - National registration number<br />ESFC...,NaN,20,20,0,0,0.37
1,23197,https://www.airbnb.com/rooms/23197,20260624162917,2026-06-25,city scrape,CCIB Forum· Large Balcony · 5 min walk to CCIB,"Beautiful, spacious apartment with large balco...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,90417,https://www.airbnb.com/users/show/90417,1.462509e+18,https://www.airbnb.com/users/profile/146250935...,Marnie,NaN,16.0,3.0,15.0,5.0,"Catalonia, Spain","Hi there,\n\nI’m marnie, originally from Austr...",NaN,NaN,NaN,t,NaN,https://a0.muscache.com/im/pictures/user/44b56...,NaN,5.0,NaN,NaN,t,t,NaN,el Besòs i el Maresme,Sant Martí,41.412432,2.21975,Entire rental unit,Entire home/apt,5,2.0,2 baths,3.0,4.0,"[""AC - split type ductless system"", ""Private p...",$388.00,2026-06-26,2026-06-30,1552.00,388.00,"{""quote"": {""taxes"": null, ""currency"": ""EUR"", ""...",2.0,1125.0,2.0,7.0,1125.0,1125.0,3.5,1125.0,NaN,t,10,34,64,239,2026-06-25,99,11,1,159,11,66,25608.0,2011-03-15,2026-06-07,4.84,4.95,4.91,4.94,4.99,4.69,4.71,Spain - National registration number<br />ESFC...,NaN,1,1,0,0,0.53
2,34981,https://www.airbnb.com/rooms/34981,20260624162917,2026-06-25,city scrape,VIDRE HOME PLAZA REAL on LAS RAMBLAS,Spacious apartment for large families or group...,NaN,https://a0.muscache.com/pictures/c4d1723c-e479...,73163,https://www.airbnb.com/users/show

In [9]:
reviews.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,6088098,37494155,2015-07-07,306060,Barton,Marta met me at the door and was very welcomin...
1,18956901,162018938,2017-06-19,72281018,Rainer,We spent our exiting trip in bcn with a chilli...
2,18956901,169085884,2017-07-11,15265416,Estrella,Javier es un excelente anfitrión. En todo mome...
3,18956901,172187716,2017-07-20,98362661,Wojciech,"Javier is a great host, easy-going person, kno..."
4,18956901,174791715,2017-07-27,137372016,Eugenia,"Javier è stato molto disponibile e gentile,ci ..."


What does one row represent?
In listings, each row represents one Airbnb property listing in Barcelona, with information about the host, location, property type, pricing, availability and review scores.

In reviews, each row represents a single review left for a listing. A listing can therefore appear many times in this table, linked back to listings through listing_id.

---
## 3. Data quality

Missing values, duplicates, numbers stored as text, dates stored as strings. Write down what you find — this is the to-do list for notebook 02.

In [10]:
listing_missing = pd.DataFrame({
    "missing_count": listings.isnull().sum(),
    "missing_percent": (listings.isnull().mean() * 100).round(2)
})

listing_missing = listing_missing[listing_missing["missing_count"] > 0]
listing_missing.sort_values("missing_percent", ascending=False)

,missing_count,missing_percent
neighborhood_overview,15293,100.00
host_since,15293,100.00
host_verifications,15293,100.00
host_total_listings_count,15293,100.00
host_thumbnail_url,15293,100.00
host_response_time,15293,100.00
host_acceptance_rate,15293,100.00
host_response_rate,15293,100.00
host_neighbourhood,15293,100.00
instant_bookable,15293,100.00


In [11]:
review_missing = pd.DataFrame({
    "missing_count": reviews.isnull().sum(),
    "missing_percent": (reviews.isnull().mean() * 100).round(2)})

review_missing = review_missing[review_missing["missing_count"] > 0]
review_missing.sort_values("missing_percent", ascending=False)

,missing_count,missing_percent
comments,114,0.01
reviewer_name,2,0.00


In [12]:
listing_missing = pd.DataFrame({
    "missing_count": listings.isnull().sum(),
    "missing_percent": (listings.isnull().mean() * 100).round(2)})

listing_missing = listing_missing[listing_missing["missing_count"] > 0]

listing_missing.sort_values("missing_percent", ascending=False)

,missing_count,missing_percent
neighborhood_overview,15293,100.00
host_since,15293,100.00
host_verifications,15293,100.00
host_total_listings_count,15293,100.00
host_thumbnail_url,15293,100.00
host_response_time,15293,100.00
host_acceptance_rate,15293,100.00
host_response_rate,15293,100.00
host_neighbourhood,15293,100.00
instant_bookable,15293,100.00


In [13]:
empty_listing_columns = listings.columns[ listings.isnull().all()]

empty_listing_columns

Index(['neighborhood_overview', 'host_since', 'host_response_time',
       'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url',
       'host_neighbourhood', 'host_total_listings_count', 'host_verifications',
       'neighbourhood', 'calendar_updated', 'instant_bookable'],
      dtype='str')

In [14]:
review_missing = pd.DataFrame({
    "missing_count": reviews.isnull().sum(),
    "missing_percent": (reviews.isnull().mean() * 100).round(2)})

review_missing = review_missing[
    review_missing["missing_count"] > 0]

review_missing.sort_values(
    "missing_percent",
    ascending=False)

,missing_count,missing_percent
comments,114,0.01
reviewer_name,2,0.00


In [15]:
reviews.columns[
    reviews.isnull().all()]

Index([], dtype='str')

The listings dataset contains a noticeable amount of missing data. Twelve columns are completely empty, so these are unlikely to be useful and can probably be removed during the cleaning stage.

Some fields that may be relevant to the analysis also contain missing values. Around 12.7% of listings have no price, while approximately 23% have no review-score information. Bedrooms and beds are missing for around 17% and 16% of listings respectively.

The missing review data may not necessarily be a data-quality error. Listings without reviews would naturally have no review score, so this should be checked before deciding how those values are handled.

The reviews dataset is much cleaner. Only 114 comments and 2 reviewer names are missing from more than one million reviews, so missing data in this table is unlikely to have a meaningful impact on the analysis.

In [16]:
listings[[
    "price",
    "last_scraped",
    "price_quote_checkin_date",
    "price_quote_checkout_date",
    "calendar_last_scraped",
    "first_review",
    "last_review"]].dtypes

price                        str
last_scraped                 str
price_quote_checkin_date     str
price_quote_checkout_date    str
calendar_last_scraped        str
first_review                 str
last_review                  str
dtype: object

In [17]:
reviews[["date"]].dtypes

date    str
dtype: object

Data types:
price is currently stored as text because the values include a currency symbol, so it will need to be converted to a numeric field before any price analysis.

Several date fields, including review dates, are also stored as text. These will need to be converted to datetime values so they can be sorted, filtered and analysed over time.

In [18]:
listings["price"].value_counts(dropna=False).head(20)

price
NaN        1938
$38.38       93
$147.00      70
$146.08      52
$38.56       48
$160.00      34
$260.00      33
$107.28      32
$320.00      29
$220.00      28
$168.58      28
$286.00      27
$215.00      27
$194.00      26
$230.00      26
$200.00      26
$263.00      25
$300.00      24
$250.00      24
$197.00      24
Name: count, dtype: int64

In [19]:
price_pattern = r"^\$[\d,]+\.\d{2}$"

unusual_prices = listings[
    listings["price"].notna() &
   ~listings["price"].str.match(price_pattern)]

len(unusual_prices)

0

Checking for price format consistancy. No values returned because all met the criteria.

note for self. used regex here as it seemed most effective but needed to look up again how to construct. Revisit this to learn more fully

In [20]:
reviews["date"].head(10)

0    2015-07-07
1    2017-06-19
2    2017-07-11
3    2017-07-20
4    2017-07-27
5    2017-09-19
6    2017-09-27
7    2018-04-29
8    2018-06-17
9    2018-07-04
Name: date, dtype: str

In [21]:
date_pattern = r"^\d{4}-\d{2}-\d{2}$"

unusual_review_dates = reviews[
    ~reviews["date"].str.match(date_pattern)]

len(unusual_review_dates)

0

As above - revisit Regex

In [22]:
listings["first_review"].head(10)

0    2013-05-27
1    2011-03-15
2    2010-10-03
3    2011-09-28
4    2011-06-16
5    2011-08-12
6    2011-06-23
7    2011-03-27
8    2011-05-08
9    2014-08-19
Name: first_review, dtype: str

In [23]:
unusual_first_review = listings[
    listings["first_review"].notna() &
    ~listings["first_review"].str.match(date_pattern)]

len(unusual_first_review)

0

In [24]:
listings["last_review"].head(10)

0    2026-05-28
1    2026-06-07
2    2026-05-26
3    2023-12-13
4    2026-06-19
5    2026-01-02
6    2011-06-23
7    2026-06-08
8    2024-05-31
9    2026-06-20
Name: last_review, dtype: str

In [25]:
unusual_last_review = listings[
    listings["last_review"].notna() &
    ~listings["last_review"].str.match(date_pattern)]

len(unusual_last_review)

0

In [26]:
listings["last_scraped"].head(10)

0    2026-06-25
1    2026-06-25
2    2026-06-25
3    2026-07-03
4    2026-06-25
5    2026-07-03
6    2026-06-25
7    2026-06-25
8    2026-06-25
9    2026-06-24
Name: last_scraped, dtype: str

In [27]:
unusual_last_scraped = listings[
    listings["last_scraped"].notna() &
    ~listings["last_scraped"].str.match(date_pattern)]

len(unusual_last_scraped)

0

Formats:
Price values are stored as text but use a consistent currency format. The key date fields checked are also stored as text but consistently follow the YYYY-MM-DD format. No unusual formats were found. These data types will need to be corrected during the later cleaning stage.

In [28]:
listings.duplicated().sum()

np.int64(0)

In [29]:
reviews.duplicated().sum()

np.int64(0)

In [30]:
listings["id"].duplicated().sum()

np.int64(0)

In [31]:
reviews["id"].duplicated().sum()

np.int64(0)

Duplicates and IDs: 
No exact duplicate rows were found in either dataset. listings.id is unique across listings, and reviews.id is unique across reviews, so both behave as expected as record identifiers.

---
## 4. Distributions

The numeric columns you care about, and the outliers you will have to decide about.

In [32]:
listings["accommodates"].describe()

count    15293.000000
mean         3.689270
std          2.446062
min          1.000000
25%          2.000000
50%          3.000000
75%          5.000000
max         16.000000
Name: accommodates, dtype: float64

Accommodates: Listings typically accommodate around 3 guests, with most falling between 2 and 5. The maximum capacity is 16, suggesting a smaller number of larger properties.

In [33]:
listings["room_type"].value_counts()

room_type
Entire home/apt    10844
Private room        4262
Shared room          119
Hotel room            68
Name: count, dtype: int64

Room type: Most Barcelona listings are entire homes or apartments, with private rooms making up most of the rest. Shared rooms and hotel rooms are rare. There are only four room types in total, which makes this a simple, useful category to carry forward when we build the database.

In [34]:
(listings["room_type"].value_counts(normalize=True) * 100).round(2)

room_type
Entire home/apt    70.91
Private room       27.87
Shared room         0.78
Hotel room          0.44
Name: proportion, dtype: float64

Room type (further): The Barcelona Airbnb market is dominated by entire homes and apartments, which make up just over 70% of listings. Private rooms account for just under 28%, while shared rooms and hotel rooms are rare. This gives us a clear split to use later when comparing price and guest activity by accommodation type.

In [35]:
listings["neighbourhood_group_cleansed"].value_counts()

neighbourhood_group_cleansed
Eixample               5835
Ciutat Vella           3247
Sants-Montjuïc         1448
Sant Martí             1436
Gràcia                 1366
Sarrià-Sant Gervasi     888
Horta-Guinardó          358
Les Corts               327
Sant Andreu             225
Nou Barris              163
Name: count, dtype: int64

District: Airbnb listings are concentrated heavily in a few parts of Barcelona. Eixample has by far the most listings, with Ciutat Vella second. Several outer districts have far fewer listings. There are only 10 district categories, so this gives a clean location level to work with later.

In [36]:
listings["neighbourhood_cleansed"].nunique()

69

Neighbourhoods: The 10 districts break down into 69 individual neighbourhoods. That gives two useful levels of location: district for a cleaner high-level comparison, and neighbourhood for more detailed analysis where it adds value.

In [37]:
listings["property_type"].nunique()

50

Property type: There are 50 different property types in the listings data. This is much more detailed than room_type, so it may be useful for deeper analysis later, but room_type is likely to be the clearer category for the main comparisons.

In [38]:
listings["property_type"].value_counts().head(10)

property_type
Entire rental unit             9772
Private room in rental unit    3021
Room in hotel                   449
Entire serviced apartment       364
Entire condo                    312
Private room in hostel          210
Entire loft                     187
Private room in home            162
Room in boutique hotel          133
Private room in condo           111
Name: count, dtype: int64

Property type: The detailed property categories are heavily concentrated in just a few types. Entire rental units and private rooms in rental units dominate the dataset, while the remaining categories are much smaller. This supports using room_type for the main comparisons and keeping property_type as a more detailed field if we need it later.

In [39]:
listings["host_id"].nunique()

4595

Hosts: There are 4,595 unique hosts across 15,293 listings, so the market clearly includes a mix of one-property hosts and hosts managing multiple listings.

In [40]:
listings["host_id"].value_counts().head(10)

host_id
346367515    588
1447144      448
21726991     359
32037490     293
4459553      243
221480824    243
299462       140
36607755     136
265193861    125
158023606    120
Name: count, dtype: int64

Host concentration: Although there are 4,595 unique hosts, some manage very large numbers of listings. The largest host in the dataset has 588 listings, with several others managing well over 100. This suggests that Airbnb supply is not evenly spread across hosts and is worth looking at in more detail.

In [41]:
host_listing_counts = listings["host_id"].value_counts()

(host_listing_counts > 1).sum()

np.int64(1607)

Multi-listing hosts: 1,607 hosts manage more than one property, which is around 35% of all hosts in the dataset. This group accounts for a significant share of the available supply, so it makes sense to look at how that concentration varies across Barcelona.

In [42]:
multi_host_ids = host_listing_counts[host_listing_counts > 1].index

listings["host_id"].isin(multi_host_ids).mean() * 100

np.float64(80.46164912051266)

Host concentration: Multi-listing hosts make up around 35% of hosts, but they control just over 80% of all listings. That suggests the Barcelona Airbnb market is heavily weighted towards hosts operating multiple properties rather than one-off individual hosts.

In [43]:
listings["review_scores_rating"].describe()

count    11745.000000
mean         4.617601
std          0.490382
min          1.000000
25%          4.500000
50%          4.730000
75%          4.920000
max          5.000000
Name: review_scores_rating, dtype: float64

Review ratings: Ratings are generally very high. The median rating is 4.73 out of 5, and three quarters of rated listings score at least 4.50. There are a small number of much lower scores, but these are not typical of the dataset. 

In [44]:
listings["number_of_reviews"].describe()

count    15293.000000
mean        67.150396
std        132.909640
min          0.000000
25%          1.000000
50%          9.000000
75%         77.000000
max       2740.000000
Name: number_of_reviews, dtype: float64

Number of reviews: Review activity is very uneven across listings. The median listing has only 9 reviews, while a small number have hundreds or even thousands. The maximum is 2,740 reviews, which pulls the average up to 67 and shows that the distribution is strongly weighted towards a relatively small number of heavily reviewed listings.

In [45]:
(listings["number_of_reviews"] == 0).sum()

np.int64(3548)

In [46]:
pd.crosstab(
    listings["number_of_reviews"] == 0,
    listings["review_scores_rating"].isna(),
    rownames=["No reviews"],
    colnames=["Rating missing"]
)

Rating missing,False,True
No reviews,,
False,11745,0
True,0,3548


Missing review scores: The missing rating values are explained by review history rather than bad data. All 3,548 listings with no reviews also have no rating, while every listing with at least one review has a rating. These nulls therefore carry useful information and should not automatically be treated as errors.

In [47]:
listings["availability_365"].describe()

count    15293.000000
mean       212.434970
std        119.050634
min          0.000000
25%        116.000000
50%        240.000000
75%        319.000000
max        365.000000
Name: availability_365, dtype: float64

Availability: Availability varies a lot across listings. The median listing is available for 240 days of the year, but the range runs from 0 to 365 days. This suggests a mix of listings with limited availability and others that appear to be available for most or all of the year.

In [48]:
listings["estimated_occupancy_l365d"].describe()

count    15293.000000
mean        86.547636
std         95.308260
min          0.000000
25%          0.000000
50%         54.000000
75%        168.000000
max        255.000000
Name: estimated_occupancy_l365d, dtype: float64

Estimated occupancy: Estimated occupancy varies widely across listings. A quarter of listings show no estimated occupied days at all, while the median is 54 days and some listings reach more than 250. This suggests a very mixed market, with some listings showing little apparent activity and others operating much more intensively.

In [49]:
listings["estimated_revenue_l365d"].describe()

count     13355.000000
mean      21878.952602
std       30151.918054
min           0.000000
25%           0.000000
50%        8268.000000
75%       35218.000000
max      440130.000000
Name: estimated_revenue_l365d, dtype: float64

Estimated revenue: Revenue is very uneven across listings. The median is just over 8,000, while the average is much higher at around 21,900 because a small number of listings generate very high estimated revenue. A quarter of listings show no estimated revenue at all.

In [50]:
listings["minimum_nights"].describe()

count    15292.000000
mean        14.058920
std         21.359699
min          1.000000
25%          1.000000
50%          2.000000
75%         31.000000
max        900.000000
Name: minimum_nights, dtype: float64

Minimum nights: Most listings have a short minimum stay, with a median of 2 nights. However, the distribution has a long upper tail: 25% of listings require at least 31 nights, and the maximum is 900. These larger values are worth checking later to understand whether they represent genuine long-stay listings or unusual cases.

In [51]:
listings["minimum_nights"].value_counts().head(10)

minimum_nights
1.0     5433
32.0    2647
31.0    2644
2.0     2539
3.0      892
4.0      230
25.0     207
30.0     146
5.0      115
60.0      55
Name: count, dtype: int64

Minimum stay pattern: Most listings allow very short stays, especially 1–2 nights. However, there is also a large cluster at 31–32 nights, showing that a significant part of the market is set up for longer stays. This helps explain why the average minimum stay is much higher than the median.

In [52]:
listings.select_dtypes(include="str").nunique().sort_values()

source                              2
host_identity_verified              2
host_has_profile_pic                2
host_is_superhost                   2
has_availability                    2
last_scraped                        4
room_type                           4
calendar_last_scraped               4
neighbourhood_group_cleansed       10
bathrooms_text                     37
property_type                      50
neighbourhood_cleansed             69
price_quote_checkin_date          293
host_location                     321
price_quote_checkout_date         332
last_review                      1594
host_about                       2406
host_name                        2510
first_review                     3836
host_picture_url                 4468
host_profile_url                 4555
host_url                         4595
price                            6365
license                         10371
description                     11815
price_quote_raw                 12192
amenities   

Categorical cardinality: Some text fields contain only a small number of repeated categories, while others are effectively unique identifiers or descriptions. room_type has 4 categories, there are 10 districts and 69 neighbourhoods, and property_type has 50 values. The district and neighbourhood fields give us a useful two-level location structure for the database later.

In [53]:
reviews.select_dtypes(include="str").nunique().sort_values()

date               5217
reviewer_name    126356
comments         985107
dtype: int64

Reviews cardinality: The review table contains mostly high-cardinality text fields. Dates repeat, but reviewer names and especially comments are highly varied, so they are not useful candidates for lookup tables. The main structure in this table is the link back to each listing through listing_id.

**IMPORTANT** - A useful limitation to note is that the occupancy and revenue figures are estimates rather than direct booking data. Inside Airbnb uses review activity as part of its occupancy model, converting reviews into estimated bookings before estimating occupied nights and income. This means these fields are not fully independent of the review data and should be interpreted carefully if reviews are also being used as an explanatory measure. - source: Inside Airbnb https://insideairbnb.com/data-assumptions/

---
## 5. Categorical columns

**This is the section that feeds tomorrow.** A column with few distinct values repeated over many rows is a lookup table waiting to happen. Find them.

Categorical cardinality

The listings data contains a mix of simple repeated categories and highly unique text fields. room_type has just 4 categories, while Barcelona is split into 10 districts and 69 neighbourhoods. property_type is more detailed, with 50 categories.

The district and neighbourhood fields give the data a useful two-level location structure, while room_type provides a simple way to compare different types of accommodation.

The reviews table contains mostly high-cardinality text fields. Reviewer names and comments are highly varied, so there are no obvious categorical lookup fields within this table. Its main structural value is the link back to each listing through listing_id. (note i completed the code queries in section 4)

## 6 Research questions

### 1. Which Barcelona districts offer the strongest nightly pricing potential for comparable Airbnb listings?

Why it matters: A potential investor needs a realistic view of what similar properties are currently asking per night before deciding which areas are worth investigating further.

Fields used:
- `price`
- `room_type`
- `accommodates`
- `neighbourhood_group_cleansed`
- `neighbourhood_cleansed`

Metrics:
- Median nightly price by district
- Median nightly price within comparable `room_type` and `accommodates` groups
- Neighbourhood-level drill-down where it adds useful detail

### 2. Which Barcelona districts show the strongest recent guest activity relative to the amount of Airbnb supply?

Why it matters: High review volume on its own may simply reflect an area with more listings, so an investor needs to know where activity appears strong relative to the amount of available supply.

Fields used:
- `reviews.id`
- `reviews.listing_id`
- `reviews.date`
- `listings.id`
- `neighbourhood_group_cleansed`

Metrics:
- Number of reviews in the most recent 12 months
- Number of listings per district
- Recent reviews per listing

Supporting guideposts:
- `estimated_occupancy_l365d`
- `estimated_revenue_l365d`

### 3. Which Barcelona districts have the highest concentration of supply controlled by hosts with multiple listings?

Why it matters: A potential investor needs to understand whether they would be entering a market dominated by individual hosts or competing with hosts already managing larger property portfolios.

Fields used:
- `host_id`
- `id` / `listing_id`
- `neighbourhood_group_cleansed`

Metrics:
- Number of listings per host
- Number and percentage of hosts with more than one listing
- Percentage of listings belonging to multi-listing hosts
- Multi-listing host concentration by district

---
## 7. Into notebook 02

The tables you are going to build, and the problems you have to fix first.

## 7. Into notebook 02

The next stage is to turn the raw Airbnb data into a smaller relational database built around the three research questions.

### Tables to build

I am going to use four related tables. The aim is to keep the structure meaningful without creating extra lookup tables that do not add anything useful to the analysis.

**locations** — one row per Barcelona neighbourhood.

- `location_id` — PK
- `neighbourhood_name`
- `district_name`

`location_id` will be created during processing. Keeping neighbourhood and district together gives me both levels of geography without needing a separate district table.

**hosts** — one row per Airbnb host.

- `host_id` — PK
- `host_name`
- `host_is_superhost`

`host_id` already exists in the raw data. Separating hosts from listings will allow me to measure how many properties each host manages and how that concentration varies across Barcelona.

**listings** — one row per Airbnb listing and the main table for the analysis.

- `listing_id` — PK
- `host_id` — FK to `hosts.host_id`
- `location_id` — FK to `locations.location_id`
- `room_type`
- `accommodates`
- `price`
- `minimum_nights`
- `number_of_reviews`
- `estimated_occupancy_l365d`
- `estimated_revenue_l365d`

This contains the main information needed to compare pricing, supply and the characteristics of comparable listings.

The estimated occupancy and revenue fields will be kept as supporting indicators rather than treated as confirmed bookings or income. They are modelled figures and will therefore be clearly identified as estimates when they are used.

**reviews** — one row per individual review.

- `review_id` — PK
- `listing_id` — FK to `listings.listing_id`
- `review_date`

The review data gives me a separate record of activity over time and can be linked directly back to each listing using `listing_id`.

### Relationships

- `hosts` and `listings` are linked using `host_id`.
- `locations` and `listings` are linked using `location_id`.
- `reviews` and `listings` are linked using `listing_id`.

This structure supports the three parts of the client analysis: pricing potential, evidence of guest activity and the competitive landscape.

### Problems to fix first

Before loading the data into SQL I need to:

- Convert `price` from text such as `$409.00` into a numeric value.
- Convert the relevant date fields into usable date values.
- Rename source IDs where necessary so the database uses clear names such as `listing_id`, `review_id` and `review_date`.
- Remove completely empty columns and fields that are not needed for the database or research questions.
- Handle missing values according to what they mean rather than automatically removing them.
- Decide how listings with missing prices should be treated in the pricing analysis.
- Check unusual values such as the 900-night minimum stay before deciding whether they are genuine or problematic.
- Create the `locations` and `hosts` tables without losing the relationship back to each listing.
- Check that all primary keys are unique and that the foreign keys do not create orphaned records.

No exact duplicate rows were found in either raw dataset, so duplicate removal is not expected to be a major cleaning issue.

The main goal of Notebook 02 is therefore to reduce the raw data to the fields I actually need, clean them consistently and build a four-table database that can answer the client questions in SQL.